# 04 - Kafka → Spark Streaming → ClickHouse (Speed Layer)
Consumer temps reel : Spark lit le flux Kafka (Decembre) et ecrit les agregations dans ClickHouse

Le dashboard Streamlit lit ClickHouse toutes les 3 secondes pour afficher les resultats live

In [2]:
import subprocess
subprocess.check_call(['pip', 'install', 'clickhouse-connect'])

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import clickhouse_connect
import pandas as pd

spark = SparkSession.builder \
    .appName('04_Kafka_Streaming_ClickHouse') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0') \
    .getOrCreate()

# Charger la dimension locations (statique, pour enrichir le flux)
locations = spark.read.parquet('hdfs://namenode:9000/projet/silver/locations_clean/')

# Charger les stats batch depuis ClickHouse (pour comparaison / alertes)
ch_client = clickhouse_connect.get_client(host='clickhouse', port=8123)
batch_stats = ch_client.query_df("SELECT borough, nb_trips, avg_fare, avg_distance FROM batch_stats_borough")

print('Spark connecte')
print(f'Locations: {locations.count()} zones')
print(f'Stats batch chargees: {len(batch_stats)} boroughs')
print(batch_stats)

Spark connecte
Locations: 263 zones
Stats batch chargees: 6 boroughs
         borough  nb_trips  avg_fare  avg_distance
0          Bronx      8712     29.29          4.46
1       Brooklyn    154310     25.14          5.54
2            EWR       118     97.92          4.51
3      Manhattan    657023     24.61          5.15
4         Queens    178085     72.05         14.30
5  Staten Island      1752     34.35         27.85


In [3]:
# Schema des messages JSON dans Kafka (avec event_time ajoute par le producer)
trip_schema = StructType([
    StructField('date_id', DoubleType()),
    StructField('pickup_location_id', IntegerType()),
    StructField('dropoff_location_id', IntegerType()),
    StructField('weather_id', DoubleType()),
    StructField('trip_distance', DoubleType()),
    StructField('total_amount', DoubleType()),
    StructField('event_time', StringType())
])

# Lire le flux Kafka
kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka:29092') \
    .option('subscribe', 'nyc-taxi-trips') \
    .option('startingOffsets', 'earliest') \
    .load()

# Parser les messages JSON
trips_stream = kafka_df \
    .select(from_json(col('value').cast('string'), trip_schema).alias('data')) \
    .select('data.*')

print('Flux Kafka connecte')
print(f'Schema : {trip_schema.names}')

Flux Kafka connecte
Schema : ['date_id', 'pickup_location_id', 'dropoff_location_id', 'weather_id', 'trip_distance', 'total_amount', 'event_time']


In [5]:
# ══════════════════════════════════════════════════
# Stream 1 : Stats par borough en live → ClickHouse
# ══════════════════════════════════════════════════
# Enrichir avec les zones puis agreger par borough

live_borough = (trips_stream
    .join(broadcast(locations),
          trips_stream.pickup_location_id == locations.location_id,
          'left')
    .groupBy('borough')
    .agg(
        count('*').alias('nb_trips'),
        round(sum('total_amount'), 2).alias('total_revenue'),
        round(avg('trip_distance'), 2).alias('avg_distance'),
        round(avg('total_amount'), 2).alias('avg_fare')
    )
)

def write_borough_to_clickhouse(batch_df, batch_id):
    """Ecrit les stats borough dans ClickHouse (upsert via ReplacingMergeTree)"""
    if batch_df.count() == 0:
        return
    pdf = batch_df.toPandas()
    pdf = pdf.dropna(subset=['borough'])
    if pdf.empty:
        return
    client = clickhouse_connect.get_client(host='clickhouse', port=8123)
    client.command("TRUNCATE TABLE stream_borough_stats")
    client.insert_df("stream_borough_stats", pdf)
    client.close()
    print(f"  [Batch {batch_id}] Borough stats: {len(pdf)} lignes → ClickHouse")

query_borough = live_borough.writeStream \
    .outputMode('complete') \
    .foreachBatch(write_borough_to_clickhouse) \
    .trigger(processingTime='2 seconds') \
    .start()

print('Stream 1 demarre : Stats borough → ClickHouse (toutes les 2s)')

Stream 1 demarre : Stats borough → ClickHouse (toutes les 2s)


In [4]:
# ══════════════════════════════════════════════════
# Stream 2 : Top zones en live → ClickHouse
# ══════════════════════════════════════════════════

live_zones = (trips_stream
    .join(broadcast(locations),
          trips_stream.pickup_location_id == locations.location_id,
          'left')
    .groupBy('zone', 'borough')
    .agg(count('*').alias('nb_trips'))
)

def write_zones_to_clickhouse(batch_df, batch_id):
    """Ecrit les top zones dans ClickHouse"""
    if batch_df.count() == 0:
        return
    pdf = batch_df.toPandas()
    pdf = pdf.dropna(subset=['zone'])
    if pdf.empty:
        return
    client = clickhouse_connect.get_client(host='clickhouse', port=8123)
    client.command("TRUNCATE TABLE stream_top_zones")
    client.insert_df("stream_top_zones", pdf)
    client.close()
    print(f"  [Batch {batch_id}] Top zones: {len(pdf)} lignes → ClickHouse")

query_zones = live_zones.writeStream \
    .outputMode('complete') \
    .foreachBatch(write_zones_to_clickhouse) \
    .trigger(processingTime='3 seconds') \
    .start()

print('Stream 2 demarre : Top zones → ClickHouse (toutes les 3s)')

Stream 2 demarre : Top zones → ClickHouse (toutes les 3s)


In [5]:
# ══════════════════════════════════════════════════
# Stream 3 : Detection d'anomalies temps reel
# ══════════════════════════════════════════════════
# Compare les stats streaming vs batch (historique)
# Si le tarif moyen ou la distance devie de > 30%, genere une alerte

live_anomaly = (trips_stream
    .join(broadcast(locations),
          trips_stream.pickup_location_id == locations.location_id,
          'left')
    .groupBy('borough')
    .agg(
        count('*').alias('nb_trips'),
        round(avg('total_amount'), 2).alias('avg_fare'),
        round(avg('trip_distance'), 2).alias('avg_distance')
    )
)

def detect_anomalies(batch_df, batch_id):
    """Compare streaming vs batch et ecrit les alertes dans ClickHouse"""
    if batch_df.count() == 0 or batch_stats.empty:
        return
    
    pdf_stream = batch_df.toPandas()
    pdf_stream = pdf_stream.dropna(subset=['borough'])
    if pdf_stream.empty:
        return
    
    alerts = []
    import datetime
    now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    for _, row in pdf_stream.iterrows():
        borough = row['borough']
        batch_row = batch_stats[batch_stats['borough'] == borough]
        if batch_row.empty:
            continue
        
        batch_fare = batch_row.iloc[0]['avg_fare']
        stream_fare = row['avg_fare']
        
        if batch_fare > 0:
            deviation = abs(stream_fare - batch_fare) / batch_fare * 100
            if deviation > 30:
                alerts.append({
                    'alert_time': now,
                    'alert_type': 'fare_deviation',
                    'borough': borough,
                    'message': f'Tarif moyen {stream_fare}$ vs historique {batch_fare}$ ({deviation:.0f}% deviation)',
                    'current_value': float(stream_fare),
                    'historical_value': float(batch_fare),
                    'deviation_pct': round(float(deviation), 1)
                })
        
        batch_dist = batch_row.iloc[0]['avg_distance']
        stream_dist = row['avg_distance']
        
        if batch_dist > 0:
            deviation_dist = abs(stream_dist - batch_dist) / batch_dist * 100
            if deviation_dist > 30:
                alerts.append({
                    'alert_time': now,
                    'alert_type': 'distance_deviation',
                    'borough': borough,
                    'message': f'Distance moy {stream_dist}mi vs historique {batch_dist}mi ({deviation_dist:.0f}% deviation)',
                    'current_value': float(stream_dist),
                    'historical_value': float(batch_dist),
                    'deviation_pct': round(float(deviation_dist), 1)
                })
    
    if alerts:
        client = clickhouse_connect.get_client(host='clickhouse', port=8123)
        client.insert_df("stream_alerts", pd.DataFrame(alerts))
        client.close()
        print(f"  [Batch {batch_id}] {len(alerts)} alertes detectees!")

query_anomaly = live_anomaly.writeStream \
    .outputMode('complete') \
    .foreachBatch(detect_anomalies) \
    .trigger(processingTime='3 seconds') \
    .start()

print('Stream 3 demarre : Detection anomalies (toutes les 3s)')

Stream 3 demarre : Detection anomalies (toutes les 3s)
  [Batch 0] Top zones: 83 lignes → ClickHouse
  [Batch 1] Top zones: 97 lignes → ClickHouse
  [Batch 2] Top zones: 110 lignes → ClickHouse
  [Batch 3] Top zones: 110 lignes → ClickHouse
  [Batch 4] Top zones: 113 lignes → ClickHouse
  [Batch 5] Top zones: 119 lignes → ClickHouse
  [Batch 6] Top zones: 125 lignes → ClickHouse
  [Batch 7] Top zones: 128 lignes → ClickHouse
  [Batch 8] Top zones: 129 lignes → ClickHouse
  [Batch 9] Top zones: 129 lignes → ClickHouse
  [Batch 10] Top zones: 129 lignes → ClickHouse
  [Batch 11] Top zones: 134 lignes → ClickHouse
  [Batch 12] Top zones: 134 lignes → ClickHouse
  [Batch 13] Top zones: 137 lignes → ClickHouse
  [Batch 14] Top zones: 137 lignes → ClickHouse
  [Batch 15] Top zones: 146 lignes → ClickHouse
  [Batch 16] Top zones: 150 lignes → ClickHouse
  [Batch 17] Top zones: 150 lignes → ClickHouse


In [6]:
# Verifier les resultats en direct (re-executez pour voir l'evolution)
print("=== Stats Borough Live ===")
ch = clickhouse_connect.get_client(host='clickhouse', port=8123)
print(ch.query_df("SELECT * FROM stream_borough_stats FINAL ORDER BY nb_trips DESC"))
print("\n=== Top 5 Zones Live ===")
print(ch.query_df("SELECT * FROM stream_top_zones FINAL ORDER BY nb_trips DESC LIMIT 5"))
print("\n=== Alertes Recentes ===")
print(ch.query_df("SELECT * FROM stream_alerts ORDER BY alert_time DESC LIMIT 5"))
ch.close()

=== Stats Borough Live ===
         borough  nb_trips  total_revenue  avg_distance  avg_fare  \
0      Manhattan    166494     4148898.31          4.68     24.92   
1         Queens     75723     4029666.11          9.53     53.22   
2       Brooklyn     33280     1036847.52          3.39     31.16   
3          Bronx       890       30716.91        111.78     34.51   
4  Staten Island       314       13786.06          5.13     43.90   
5            EWR       139       13498.96          0.70     97.11   

           updated_at  
0 2026-05-28 17:46:29  
1 2026-05-28 17:46:29  
2 2026-05-28 17:46:29  
3 2026-05-28 17:46:29  
4 2026-05-28 17:46:29  
5 2026-05-28 17:46:29  

=== Top 5 Zones Live ===
              zone    borough  nb_trips          updated_at
0         Union Sq  Manhattan     38766 2026-05-28 18:42:43
1    Melrose South      Bronx     37450 2026-05-28 18:42:43
2          Jamaica     Queens     37079 2026-05-28 18:42:43
3   Middle Village     Queens     30147 2026-05-28 18:4

In [7]:
# Arreter tous les streams
query_borough.stop()
query_zones.stop()
query_anomaly.stop()
print('Tous les streams arretes')

ch_client.close()
spark.stop()
print('Connexions fermees')

NameError: name 'query_zones' is not defined